# Telecom Customer Churn Analysis and Prediction

**Student:** Bharat Goyal  
**Internship:** IBM SkillsBuild Data Analytics with AI Academic Internship  
**Organization:** BharatCares in association with AICTE

### Objective
Analyze telecom customer data to identify factors associated with customer churn and build a machine-learning model that estimates churn risk.

> **Note:** The notebook is intentionally self-contained. It first looks for the public IBM Telco dataset locally, then attempts an automatic download. If neither is possible, it generates a clearly labelled synthetic dataset so the notebook remains runnable for demonstration.

## 1. Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42


## 2. Load the Dataset

In [ ]:
DATA_FILE = "WA_FnUseC_TelcoCustomerChurn.csv"
DATA_URL = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cpd4.5/data/customer_churn/WA_FnUseC_TelcoCustomerChurn.csv"

def make_demo_dataset(n=2500, seed=42):
    rng = np.random.default_rng(seed)
    gender = rng.choice(["Male", "Female"], n)
    senior = rng.choice([0, 1], n, p=[0.84, 0.16])
    partner = rng.choice(["Yes", "No"], n, p=[0.48, 0.52])
    dependents = rng.choice(["Yes", "No"], n, p=[0.30, 0.70])
    tenure = rng.integers(1, 73, n)
    phone = rng.choice(["Yes", "No"], n, p=[0.90, 0.10])
    internet = rng.choice(["DSL", "Fiber optic", "No"], n, p=[0.34, 0.44, 0.22])
    contract = rng.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.21, 0.24])
    paperless = rng.choice(["Yes", "No"], n, p=[0.60, 0.40])
    payment = rng.choice(
        ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
        n, p=[0.34, 0.23, 0.22, 0.21]
    )
    monthly = np.round(
        20 + (internet == "Fiber optic")*55 + (internet == "DSL")*30 +
        rng.normal(0, 10, n) + (contract == "Month-to-month")*7, 2
    )
    monthly = np.clip(monthly, 18, 125)
    total = np.round(monthly * tenure + rng.normal(0, 30, n), 2)
    total = np.maximum(total, monthly)

    # Construct a realistic educational target with known drivers.
    logit = (
        -1.8
        + 1.15*(contract == "Month-to-month")
        + 0.65*(internet == "Fiber optic")
        + 0.55*(payment == "Electronic check")
        + 0.45*(paperless == "Yes")
        + 0.25*senior
        - 0.045*tenure
        - 0.35*(contract == "Two year")
        - 0.25*(contract == "One year")
    )
    p = 1/(1+np.exp(-logit))
    churn = np.where(rng.random(n) < p, "Yes", "No")

    df = pd.DataFrame({
        "customerID": [f"DEMO-{i:05d}" for i in range(n)],
        "gender": gender, "SeniorCitizen": senior, "Partner": partner,
        "Dependents": dependents, "tenure": tenure, "PhoneService": phone,
        "MultipleLines": rng.choice(["Yes", "No"], n),
        "InternetService": internet,
        "OnlineSecurity": rng.choice(["Yes", "No"], n),
        "OnlineBackup": rng.choice(["Yes", "No"], n),
        "DeviceProtection": rng.choice(["Yes", "No"], n),
        "TechSupport": rng.choice(["Yes", "No"], n),
        "StreamingTV": rng.choice(["Yes", "No"], n),
        "StreamingMovies": rng.choice(["Yes", "No"], n),
        "Contract": contract,
        "PaperlessBilling": paperless, "PaymentMethod": payment,
        "MonthlyCharges": monthly, "TotalCharges": total, "Churn": churn
    })
    return df

source_used = "Local IBM dataset"
if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
else:
    try:
        df = pd.read_csv(DATA_URL)
        source_used = "IBM public dataset downloaded from GitHub"
    except Exception:
        df = make_demo_dataset()
        source_used = "Synthetic demonstration dataset (internet unavailable)"

print("Data source:", source_used)
print("Shape:", df.shape)
df.head()


## 3. Data Understanding and Cleaning

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

# Convert TotalCharges to numeric because the public dataset can contain blanks.
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove customer identifier from modelling.
df_model = df.copy()
if "customerID" in df_model.columns:
    df_model = df_model.drop(columns=["customerID"])

print("\nDuplicate rows:", df_model.duplicated().sum())


## 4. Exploratory Data Analysis

In [ ]:
churn_counts = df["Churn"].value_counts()
print(churn_counts)
print("\nOverall churn rate:", round((df["Churn"] == "Yes").mean()*100, 2), "%")

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# Churn rate by contract type
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
print(contract_churn.round(2))

plt.figure(figsize=(8,5))
sns.barplot(
    data=df, x="Contract", y=df["Churn"].eq("Yes").astype(int),
    estimator=np.mean, errorbar=None
)
plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.xlabel("Contract")
plt.ylim(0, 1)
plt.show()


In [ ]:
# Churn rate by tenure groups
df["TenureGroup"] = pd.cut(
    df["tenure"], bins=[0, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
)

tenure_churn = df.groupby("TenureGroup", observed=True)["Churn"].apply(lambda x: (x=="Yes").mean()*100)
print(tenure_churn.round(2))

plt.figure(figsize=(8,5))
sns.barplot(x=tenure_churn.index, y=tenure_churn.values)
plt.title("Churn Rate by Customer Tenure")
plt.ylabel("Churn Rate (%)")
plt.xlabel("Tenure Group")
plt.xticks(rotation=15)
plt.show()


In [ ]:
# Monthly charges distribution by churn status
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.show()

# Internet service vs churn
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
print(internet_churn.round(2))

plt.figure(figsize=(8,5))
sns.barplot(
    data=df, x="InternetService", y=df["Churn"].eq("Yes").astype(int),
    estimator=np.mean, errorbar=None
)
plt.title("Churn Rate by Internet Service")
plt.ylabel("Churn Rate")
plt.xlabel("Internet Service")
plt.show()


## 5. Correlation / Numerical Overview

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
available_numeric = [c for c in numeric_cols if c in df.columns]

if available_numeric:
    plt.figure(figsize=(7,5))
    sns.heatmap(df[available_numeric].corr(), annot=True, fmt=".2f", cmap="Blues")
    plt.title("Correlation Matrix of Numerical Variables")
    plt.show()

print(df[available_numeric].describe().T)


## 6. Prepare Data for Machine Learning

In [ ]:
# Remove helper column and target from predictors.
work = df.copy()
work = work.drop(columns=["TenureGroup"], errors="ignore")
work["Churn"] = work["Churn"].map({"No": 0, "Yes": 1})

X = work.drop(columns=["Churn", "customerID"], errors="ignore")
y = work["Churn"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
print("Model training completed.")


## 7. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

metrics_df = pd.DataFrame({"Metric": list(metrics.keys()), "Score": list(metrics.values())})
metrics_df["Score"] = metrics_df["Score"].round(4)
display(metrics_df)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"], zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted No", "Predicted Yes"],
            yticklabels=["Actual No", "Actual Yes"])
plt.title("Confusion Matrix")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


## 8. Model Explainability

In [ ]:
# Extract encoded feature names and logistic regression coefficients.
pre = model.named_steps["preprocessor"]
clf = model.named_steps["classifier"]

feature_names = pre.get_feature_names_out()
coefs = clf.coef_[0]

importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs,
    "AbsoluteImportance": np.abs(coefs)
}).sort_values("AbsoluteImportance", ascending=False)

display(importance.head(15))

top = importance.head(12).sort_values("Coefficient")
plt.figure(figsize=(9,6))
plt.barh(top["Feature"], top["Coefficient"])
plt.title("Top Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()


## 9. Automated Business Insights

In [ ]:
def rate_by(column):
    return df.groupby(column)["Churn"].apply(lambda x: (x=="Yes").mean()*100).sort_values(ascending=False)

for col in ["Contract", "InternetService", "PaymentMethod"]:
    if col in df.columns:
        print(f"\nChurn rate by {col}:")
        print(rate_by(col).round(2))

print("\nGeneral insights generated from the analysis:")
print("1. Shorter-tenure customers should receive stronger onboarding and retention support.")
print("2. Contract type is an important segmentation variable; month-to-month customers should be prioritized.")
print("3. Monthly charges can be used as a pricing/value signal when designing retention offers.")
print("4. The machine-learning model can be used to rank customers by estimated churn probability.")
print("5. Retention decisions should combine model output with customer-service and business context.")


## 10. Conclusion

This project demonstrates an end-to-end data analytics and machine-learning workflow for customer churn.

The analysis shows how data cleaning, visualization, statistical summaries and classification models can be combined to support business decisions. Logistic Regression is used because it provides both a churn probability and interpretable coefficients.

### Future Improvements

- Compare Logistic Regression with Random Forest, Gradient Boosting and XGBoost.
- Tune the probability threshold according to the cost of missing a churner.
- Add SHAP-based explainability.
- Build an interactive Streamlit dashboard.
- Add time-series customer behavior when longitudinal data becomes available.
